# EDA

## Daten laden
europe-electricity-load-hourly-20192025 aus Kaggle manuell herunterladen und als csv einlesen

In [ ]:
!pip install kagglehub

In [ ]:
# Set a consistent style for all plots
import matplotlib.pyplot as plt
plt.rcParams.update({
    'axes.grid':      True,
    'grid.color':     '#DCDCDC',
    'grid.linewidth': 0.5,
    'grid.linestyle': '-',
    'axes.axisbelow': True,
    'axes.facecolor': 'white',
    'font.family':    'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.titlepad':  13,
    'axes.labelsize': 10,
    'axes.labelpad':  8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    True,
    'legend.facecolor':  'white',
    'legend.edgecolor':  '#DCDCDC',
    'legend.framealpha': 1.0,
    'legend.fontsize':   9,
})

In [ ]:
import kagglehub
import shutil
import os

dataset_link = "dsersun/europe-electricity-load-hourly-20192025"  # just the owner/dataset part
destination = "../data/raw"

cache_path = kagglehub.dataset_download(dataset_link)
#print(f"Downloaded to cache: {cache_path}")

os.makedirs(destination, exist_ok=True)

# Copy all files from cache to destination
for file in os.listdir(cache_path):
    shutil.copy(os.path.join(cache_path, file), destination)
    print(f"Copied: {file} → {destination}")

In [ ]:
import pandas as pd

# Load the dataset
file_path = "../data/raw/MHLV_2019_2025_combined.csv"
df_energy = pd.read_csv(file_path, parse_dates=['DateUTC', 'DateShort'])

In [ ]:
display(df_energy.head())
print(df_energy.info())
print(df_energy.describe())

In [ ]:
df_energy = df_energy.drop(['DateShort', 'MeasureItem', 'TimeFrom', 'TimeTo', 'Cov_ratio', 'Value_ScaleTo100'], axis=1)
df_energy = df_energy.rename(columns={'Value': 'EnergyDemand'})
df_energy = df_energy[df_energy['CountryCode'] == 'DE']
df_energy = df_energy.drop('CountryCode', axis=1)


In [ ]:
df_energy['hour'] = df_energy['DateUTC'].dt.hour
df_energy['weekday'] = df_energy['DateUTC'].dt.dayofweek
df_energy['month'] = df_energy['DateUTC'].dt.month
df_energy['is_weekend'] = (df_energy['DateUTC'].dt.weekday >= 5).astype(int)

In [ ]:
%pip install holidays

In [ ]:
# add public holidays for Germany
import pandas as pd
import holidays 

de_holidays = holidays.Germany(years=range(2019, 2026))
df_energy['is_holiday'] = df_energy['DateUTC'].dt.date.apply(lambda x: 1 if x in de_holidays else 0)

# add holiday ratio depending the number of states in Germany with a holiday on that day
def holiday_ratio(date):
    count = sum([1 for state in holidays.Germany(years=[date.year]).items() if state[0] == date])
    return count / 16   
df_energy['holiday_ratio'] = df_energy['DateUTC'].dt.date.apply(holiday_ratio)

In [ ]:
df_energy.to_csv("../data/energy_demand_de_2019_2025.csv", index=False)

In [ ]:
df_energy.describe()

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import pandas as pd   

df_energy = pd.read_csv("../data/energy_demand_de_2019_2025.csv", parse_dates=['DateUTC'])

df_monthly = df_energy.set_index('DateUTC')['EnergyDemand'].resample('ME').mean().reset_index()

# Convert dates to float for regression
df_monthly['DateNum'] = mdates.date2num(df_monthly['DateUTC'])

fig, ax = plt.subplots(figsize=(10, 2))
sns.lineplot(data=df_monthly, x='DateUTC', y='EnergyDemand', ax=ax)
plt.title('Electricity Load in Germany (2019-01-01 - 2025-09-30) - Monthly Mean')

# add trend line
sns.regplot(data=df_monthly, x='DateNum', y='EnergyDemand',
            scatter=False, color='orange', label='Trend Line', ax=ax)

#plt.ylim(0, df_monthly['EnergyDemand'].max() * 1.1)
plt.xlabel('Year')
plt.ylabel('Load (MW)')
plt.show()

## time series decomposition

* additive model - seasonal values and redisual independent on trend

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

df_energy_de = pd.read_csv("../data/energy_demand_de_2019_2025.csv", parse_dates=['DateUTC'])
df_monthly = df_energy_de.set_index('DateUTC')['EnergyDemand'].resample('ME').mean().reset_index()
df_monthly.set_index('DateUTC', inplace=True)
# Convert dates to float for regression
df_monthly['DateNum'] = mdates.date2num(df_monthly.index)

slope, intercept = np.polyfit(df_monthly['DateNum'], df_monthly['EnergyDemand'], 1) # estimate line coefficient
trend = df_monthly['DateNum'] * slope + intercept # linear trend
detrended = df_monthly['EnergyDemand'] - trend # remove the trend

plt.figure(figsize=(10, 5))
plt.plot(df_monthly['EnergyDemand'], label='Original')
plt.plot(trend, label='Trend')
plt.plot(detrended, label='Detrended')
plt.legend()
plt.xlabel('Date')
plt.ylabel('Energy Demand (MW)')
plt.show()

In [ ]:
!pip install statsmodels

In [ ]:
# decompose seasonality using seasonal_decompose
from statsmodels.tsa.seasonal import seasonal_decompose     
#df_monthly.set_index('DateUTC', inplace=True)
decomposition = seasonal_decompose(df_monthly['EnergyDemand'], model='additive', period=12) # monthly data, so period is 12 months
fig = decomposition.plot();  # use ; to avoid doubled plot. the returned figure is already plotted by the function, so we just suppress the output to avoid a second empty plot.
fig.suptitle('Additive Decomposition', fontsize=14, y=1.05)
plt.show()

In [ ]:
# decompose seasonality using seasonal_decompose with multiplicative model
from statsmodels.tsa.seasonal import seasonal_decompose     
#df_monthly.set_index('DateUTC', inplace=True)
decomposition = seasonal_decompose(df_monthly['EnergyDemand'], model='multiplicative', period=12) # monthly data, so period is 12 months
decomposition.plot();  # use ; to avoid doubled plot. the returned figure is already plotted by the function, so we just suppress the output to avoid a second empty plot.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose, STL
from scipy.fft import fft
np.random.seed(0)  # for reproducibility

stl_decomposition = STL(endog=df_monthly['EnergyDemand'], period=12, robust=True).fit()
stl_decomposition.plot();  # use ; to avoid doubled plot. the returned figure is already plotted by the function, so we just suppress the output to avoid a second empty plot.